# Weather Stress Index for commodity futures

**Question.** Does weather in production areas (consumption areas for natural gas) help forecast
commodity futures, and does a synthetic Weather Stress Index (WSI) carry more information than
the individual weather variables?

**Pipeline** (each step is a module in `src/`, run once from the project root; this notebook only reads `data/processed/`):

| Step | Module | Output |
|---|---|---|
| 1. Downloads | `download_prices`, `download_cpc`, `download_spam` | `data/raw/` |
| 2. Weekly prices, features, targets | `prices` | `prices_weekly.parquet` |
| 3. Production weights on the weather grid | `weights` | `weights.parquet` |
| 4. Weekly weather per zone | `weather` | `weather_weekly.parquet` |
| 5. Stress components (anomalies) | `components` | `components_weekly.parquet` |
| 6. Indices (CPRI, PCA) | `indices` | `indices_weekly.parquet` |
| 7. Walk-forward models M1 to M4 | `models` | run below |

In [ ]:
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT))

from config import COMMODITIES, DATA_PROCESSED, DATA_RAW
from src import models

pd.set_option("display.float_format", "{:.3f}".format)
plt.rcParams.update({"figure.figsize": (12, 4), "axes.grid": True, "grid.alpha": 0.3})

## 1. Universe

Eight futures: seven agricultural (weather acts on **supply**) and natural gas (weather acts on **demand**).
Each commodity has one or more production zones; each zone has its own growing season.

In [ ]:
pd.DataFrame([{
    "commodity": name, "ticker": s["ticker"], "crop map": s["crop"] or "population",
    "zones": ", ".join(s["zones"]), "heat threshold (C)": s["heat_t"], "frost threshold (C)": s["frost_t"],
} for name, s in COMMODITIES.items()]).set_index("commodity")

## 2. Prices and targets

Yahoo's continuous contracts jump when the front contract expires. The return of the roll day is
removed (`prices.roll_days`). Check: if the roll days are correctly located, their absolute returns
should be clearly larger than on other days.

In [ ]:
from src.prices import roll_days

prices = pd.read_parquet(DATA_RAW / "prices_daily.parquet")
check = []
for name, spec in COMMODITIES.items():
    close = prices[name].dropna()
    abs_ret = np.log(close).diff().abs()
    is_roll = abs_ret.index.isin(roll_days(close.index, spec["roll"], spec["months"]))
    check.append({"commodity": name, "mean |r| roll days": abs_ret[is_roll].mean(),
                  "mean |r| other days": abs_ret[~is_roll].mean(), "roll days": is_roll.sum()})
pd.DataFrame(check).set_index("commodity")

In [ ]:
weekly = pd.read_parquet(DATA_PROCESSED / "prices_weekly.parquet")
cum = weekly["ret_1w"].unstack("commodity").fillna(0).cumsum()
ax = np.exp(cum).plot(logy=True, title="Roll-adjusted cumulative performance (log scale)")
ax.set_ylabel("growth of 1")
plt.show()

**Targets.** Weekly return direction is notoriously hard to forecast. We therefore also look at targets
where weather may matter more:
- `target_rv`: realized volatility of next week,
- `target_big`: next week's move exceeds 2 standard deviations (measured with the volatility known today),
- `target_up` / `target_down`: same, split by direction (a supply shock should mostly produce **up** jumps).

In [ ]:
weekly.groupby("commodity")[["target_big", "target_up", "target_down", "target_rv"]].mean() \
      .rename(columns=lambda c: c.replace("target_", "freq_" if c != "target_rv" else "mean_"))

## 3. Where the weather is measured: production weights

Each 0.5° weather cell gets the share of the zone's production located in it (MapSPAM 2010).
Zones are drawn wide; cells without production get no weight.

In [ ]:
weights = pd.read_parquet(DATA_PROCESSED / "weights.parquet")
pairs = list(weights.groupby(["commodity", "zone"]).groups)
ncols = 4
fig, axes = plt.subplots(int(np.ceil(len(pairs) / ncols)), ncols, figsize=(16, 3.2 * np.ceil(len(pairs) / ncols)))
for ax, (commodity, zone) in zip(axes.flat, pairs):
    w = weights.query("commodity == @commodity and zone == @zone")
    ax.scatter(w["lon"], w["lat"], c=w["weight"], s=6, cmap="YlOrRd")
    ax.set_title(f"{commodity} / {zone}", fontsize=10)
    ax.set_aspect("equal")
for ax in axes.flat[len(pairs):]:
    ax.axis("off")
plt.tight_layout()
plt.show()

In [ ]:
# Share of each zone in the commodity's production (used to add zones together)
mass = weights.groupby(["commodity", "zone"])["mass"].sum()
(mass / mass.groupby(level="commodity").transform("sum")).rename("share").to_frame()

## 4. Weekly weather

Daily variables are computed cell by cell, then averaged with the production weights, then summed
or averaged over the week. Example: extreme degree days (heat above 29°C) over the US corn belt.
The 2012 drought should stand out.

In [ ]:
weather = pd.read_parquet(DATA_PROCESSED / "weather_weekly.parquet")
edd = (weather.query("commodity == 'corn' and variable == 'edd'")
       .set_index("date")["value"])
by_year = edd.groupby([edd.index.year, edd.index.isocalendar().week.values]).sum().unstack(0)
ax = by_year.loc[18:40].plot(color="lightgrey", legend=False, title="US corn belt: weekly extreme degree days, weeks 18 to 40")
for year, color in [(2012, "red"), (2023, "orange"), (2014, "blue")]:
    if year in by_year:
        by_year.loc[18:40, year].plot(ax=ax, color=color, lw=2, label=str(year))
ax.legend(); ax.set_xlabel("week of year")
plt.show()

## 5. Stress components and indices

**Components** are anomalies (z-scores) of each weather variable against the same weeks of the
year in *previous* years only, set to 0 outside the growing season, then summed over zones with
production shares.

**Indices**
- `wsi_cpri`: components min-max scaled with their past range, equal weights (Guo, Ji and Zhang, 2024),
- `wsi_pca`: first principal component, refit each January on past weeks (Ma, Zhou and Li, 2024).

In [ ]:
components = pd.read_parquet(DATA_PROCESSED / "components_weekly.parquet")
indices = pd.read_parquet(DATA_PROCESSED / "indices_weekly.parquet")
components.groupby("commodity").std().round(2)

In [ ]:
EVENTS = {"corn": ["2012-07-15"], "soybeans": ["2012-08-01"], "coffee": ["2021-07-20"],
          "cocoa": ["2023-12-01"], "natgas": ["2021-02-15"], "wheat_srw": ["2010-07-15"],
          "wheat_hrw": ["2011-05-01", "2022-05-01"]}

names = indices.index.get_level_values("commodity").unique()
fig, axes = plt.subplots(len(names), 1, figsize=(14, 2.6 * len(names)), sharex=True)
for ax, name in zip(np.atleast_1d(axes), names):
    idx = indices.xs(name, level="commodity")["wsi_pca"].loc["2000":]
    ax.plot(idx.index, idx.values, lw=0.7, label="WSI-PCA")
    price = weekly.xs(name, level="commodity")["ret_1w"].fillna(0).cumsum().loc["2000":]
    ax2 = ax.twinx(); ax2.plot(price.index, price.values, color="black", lw=0.6, alpha=0.6); ax2.grid(False)
    for date in EVENTS.get(name, []):
        ax.axvline(pd.Timestamp(date), color="red", ls="--", lw=1)
    ax.set_title(f"{name}: WSI-PCA (blue) and cumulative log return (black)", fontsize=10)
plt.tight_layout()
plt.show()

In [ ]:
# What drives the PCA index? Loadings re-estimated every year (expanding window)
loadings = pd.read_parquet(DATA_PROCESSED / "pca_loadings.parquet")
fig, axes = plt.subplots(1, len(names), figsize=(4 * len(names), 4), sharey=True)
for ax, name in zip(np.atleast_1d(axes), names):
    l = loadings.query("commodity == @name").set_index("year").drop(columns=["commodity"])
    l = l.dropna(axis=1, how="all")
    shown = l.drop(columns="explained_var")
    ax.imshow(shown.T.values, aspect="auto", cmap="RdBu_r", vmin=-1, vmax=1)
    ax.set_yticks(range(len(shown.columns)), shown.columns)
    ax.set_xticks(range(0, len(shown), 5), shown.index[::5])
    ax.set_title(f"{name}\nexpl. var {l['explained_var'].iloc[-1]:.0%}", fontsize=10)
plt.tight_layout()
plt.show()

## 6. Forecasting: M1 to M4

Walk-forward: refit every January from 2010, train on all previous weeks.
`ridge` = linear benchmark (logistic for binary targets), `boosting` = shallow gradient-boosted trees.

Read the tables row by row: the question is whether M2, M3, M4 beat M1, and whether M3 (index)
beats M2 (individual variables).

In [ ]:
TARGETS = ["target_1w", "target_rv", "target_big", "target_up"]
results = {}
for target in TARGETS:
    scores, preds = models.run_all(target, panel=True)
    results[target] = (scores, preds)
    print(target)
    display(scores.drop(columns="group").set_index(["model", "features"]))

In [ ]:
# Same, one model per commodity, for the most promising binary target
scores, _ = models.run_all("target_up", panel=False, kinds=("ridge",))
scores.pivot(index="group", columns="features", values="auc")

## 7. Next steps

- Backtest (`backtest.py`): position sizing from forecasts, costs, Sharpe.
- Interpretation: SHAP on the boosting model, results in season vs out of season.
- Robustness: one-week weather lag, no season mask, sub-periods.
- Extensions: text index from the USDA Weekly Weather and Crop Bulletin, weather forecasts.